# 06 — Text-to-Image Generation

## What
Text-to-image models generate photorealistic or artistic images from natural language prompts. The dominant architecture is *latent diffusion*: a VAE compresses images to a compact latent space, and a diffusion model operates in that latent space conditioned on text embeddings from CLIP or T5.

## Why
Generating in latent space (instead of pixel space) reduces compute by 64x for typical 512×512 images with 8x downsampling. Text conditioning via cross-attention at every UNet block lets the model attend to relevant text tokens when generating each spatial region.

## Community context
DALL-E 2 (2022) used CLIP embeddings + diffusion. Stable Diffusion (Rombach et al., 2022) was the first open-source latent diffusion model — it democratised image generation. DALL-E 3 (2023) added improved caption quality via recaptioning with GPT-4V. SDXL, Flux, and PixArt-alpha pushed quality further. Imagen (Google) used large language models (T5-XXL) as text encoders, showing that better text understanding directly improves generation quality.

In [ ]:
# Text-conditioned latent diffusion — forward/reverse pass skeleton
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

class TextEncoder:
    """Simulates CLIP text encoder: prompt -> (seq_len, text_dim)"""
    def __init__(self, vocab_size=100, text_dim=64, max_len=77):
        self.embed = np.random.randn(vocab_size, text_dim) * 0.1
        self.max_len = max_len
        self.text_dim = text_dim

    def encode(self, prompt_tokens):
        # Pad/truncate to max_len
        tokens = prompt_tokens[:self.max_len]
        pad = self.max_len - len(tokens)
        embeds = np.array([self.embed[t] for t in tokens])
        if pad > 0:
            embeds = np.vstack([embeds, np.zeros((pad, self.text_dim))])
        return embeds  # (77, 64)

class CrossAttentionBlock:
    """Latent spatial features attend to text context"""
    def __init__(self, spatial_dim, text_dim, n_heads=4):
        scale = np.sqrt(spatial_dim)
        self.Wq = np.random.randn(spatial_dim, spatial_dim) / scale
        self.Wk = np.random.randn(text_dim, spatial_dim)    / scale
        self.Wv = np.random.randn(text_dim, spatial_dim)    / scale
        self.head_dim = spatial_dim // n_heads

    def __call__(self, spatial_feats, text_feats):
        """spatial_feats: (H*W, spatial_dim), text_feats: (77, text_dim)"""
        Q = spatial_feats @ self.Wq   # (HW, D)
        K = text_feats    @ self.Wk   # (77, D)
        V = text_feats    @ self.Wv
        attn = softmax(Q @ K.T / np.sqrt(self.head_dim))
        return spatial_feats + attn @ V  # residual

class SimpleLatentDiffusionUNet:
    """
    Minimal U-Net block: takes (latent_hw, latent_dim) + text context
    and predicts the noise epsilon in the latent.
    """
    def __init__(self, latent_dim=32, text_dim=64, n_heads=4):
        # Timestep embedding (sinusoidal)
        self.t_dim = latent_dim
        # Cross-attention to text
        self.cross_attn = CrossAttentionBlock(latent_dim, text_dim, n_heads)
        # Noise prediction head
        self.W_eps = np.random.randn(latent_dim, latent_dim) * 0.1

    def timestep_embedding(self, t, dim):
        half = dim // 2
        freqs = np.exp(-np.arange(half) * np.log(10000) / half)
        emb = np.concatenate([np.sin(t*freqs), np.cos(t*freqs)])
        return emb

    def __call__(self, z_noisy, t, text_context):
        """z_noisy: (HW, D), t: int, text_context: (77, text_dim)"""
        t_emb = self.timestep_embedding(t, self.t_dim)  # (D,)
        # Add timestep info to spatial features
        z = z_noisy + t_emb[None, :]  # broadcast over HW
        # Cross-attend to text
        z = self.cross_attn(z, text_context)
        # Predict noise
        eps_pred = z @ self.W_eps
        return eps_pred

np.random.seed(42)
text_encoder = TextEncoder()
unet = SimpleLatentDiffusionUNet(latent_dim=32, text_dim=64)

# Simulate: prompt='a red cat on a beach'
prompt_tokens = [12, 45, 7, 33, 8, 92]  # token IDs
text_ctx = text_encoder.encode(prompt_tokens)    # (77, 64)

# Latent: 8x8 spatial grid (corresponds to 64x64 image with 8x VAE downsampling)
HW = 64  # 8x8
z_T = np.random.randn(HW, 32)  # pure noise at T=1000

print('Latent Diffusion UNet forward pass:')
print(f'  Noisy latent z_T:   {z_T.shape}')
print(f'  Text context:       {text_ctx.shape}')

# One denoising step
for t in [1000, 500, 100, 1]:
    eps = unet(z_T, t, text_ctx)
    print(f'  t={t:4d}: eps_pred shape={eps.shape} mean={eps.mean():.4f}')

## Classifier-Free Guidance in Text-to-Image

CFG steers generation toward the text prompt by amplifying the difference between text-conditioned and unconditioned noise predictions:

```
eps_guided = eps_uncond + guidance_scale * (eps_text - eps_uncond)
```

Higher guidance scale = stronger prompt adherence, lower diversity.

In [ ]:
# CFG in text-to-image context
def cfg_step(unet, z_noisy, t, text_ctx, null_ctx, guidance_scale=7.5):
    eps_text   = unet(z_noisy, t, text_ctx)   # conditioned
    eps_uncond = unet(z_noisy, t, null_ctx)   # unconditioned (null prompt)
    eps_guided = eps_uncond + guidance_scale * (eps_text - eps_uncond)
    return eps_guided

null_ctx = np.zeros_like(text_ctx)  # empty prompt embedding

print('CFG guidance scale ablation:')
print(f'{"Scale":<8} {"Norm of guided eps":>20} {"Deviation from uncond":>22}')
for scale in [1.0, 3.0, 7.5, 15.0, 30.0]:
    eps_g = cfg_step(unet, z_T, 500, text_ctx, null_ctx, scale)
    eps_u = unet(z_T, 500, null_ctx)
    norm_g = np.linalg.norm(eps_g)
    dev = np.linalg.norm(eps_g - eps_u)
    print(f'{scale:<8} {norm_g:>20.4f} {dev:>22.4f}')
print('\nHigher scale: stronger prompt adherence, higher norm (more extreme noise pred)')